<a href="https://colab.research.google.com/github/natalia-maler/Housing_price_prediction_using_neural_networks_and_Random_Forest/blob/main/Housing_price_prediction_using_neural_networks_and_Random_Forest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Celem projektu było zbudowanie modeli regresyjnych przewidujących medianę cen mieszkań na podstawie cech demograficznych i geograficznych pochodzących ze zbioru California Housing. W projekcie porównano skuteczność prostego modelu sieci neuronowej, ulepszonej sieci neuronowej oraz modelu Random Forest ze strojeniem hiperparametrów.

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.set_printoptions(precision=12, suppress=True, linewidth=150) # 12 miejsc po przecinku, suppress bez notacji naukowej, linewidth szerokość wyświetlania
pd.options.display.float_format = '{:.6f}'.format
tf.__version__

'2.20.0'

In [2]:
# dane - chmura google cloud
url = "https://storage.googleapis.com/esmartdata-courses-files/ann-course/housing.csv"
df = pd.read_csv(url)
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.230000,37.880000,41.000000,880.000000,129.000000,322.000000,126.000000,8.325200,452600.000000,NEAR BAY
1,-122.220000,37.860000,21.000000,7099.000000,1106.000000,2401.000000,1138.000000,8.301400,358500.000000,NEAR BAY
2,-122.240000,37.850000,52.000000,1467.000000,190.000000,496.000000,177.000000,7.257400,352100.000000,NEAR BAY
3,-122.250000,37.850000,52.000000,1274.000000,235.000000,558.000000,219.000000,5.643100,341300.000000,NEAR BAY
4,-122.250000,37.850000,52.000000,1627.000000,280.000000,565.000000,259.000000,3.846200,342200.000000,NEAR BAY


Wyjaśnienie kolumn

Zbiór danych California Housing pochodzi z amerykańskiego spisu ludności i zawiera informacje o warunkach mieszkaniowych w różnych regionach Kalifornii.
Celem zadania jest przewidywanie mediany wartości domu (median_house_value) na podstawie cech demograficznych i geograficznych danego obszaru.

- longitude - długość geograficzna, położenie w kierunku wschód-zachód.
- latitude - szerokość geograficzna, położenie w kierunku południe-półudnie.
- housing_median_age - mediana wieku budynków w okolicy.
- total_rooms - łączna liczba pokoi w danym obszarze, a nie jednego domu.
- total_bedrooms - liczba sypialni, a nie jednego domu.
- population - liczba mieszkańców.
- households - liczba gospodarstw domowych.
- median_income - mediana dochodu gospodarstw domowych, w dziesiątkach tysięcy USD: 8.3252 ~ 83 252 USD.
- ocean_proximity - położenie względem oceanu, zmienna tekstowa np. INLAND - daleko od oceanu, NEAR BAY - blisko zatoki .
- median_house_value - wartość domu, zmienna przewidywana.


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           20640 non-null  float64
 1   latitude            20640 non-null  float64
 2   housing_median_age  20640 non-null  float64
 3   total_rooms         20640 non-null  float64
 4   total_bedrooms      20433 non-null  float64
 5   population          20640 non-null  float64
 6   households          20640 non-null  float64
 7   median_income       20640 non-null  float64
 8   median_house_value  20640 non-null  float64
 9   ocean_proximity     20640 non-null  object 
dtypes: float64(9), object(1)
memory usage: 1.6+ MB


In [4]:
df.describe()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
count,20640.000000,20640.000000,20640.000000,20640.000000,20433.000000,20640.000000,20640.000000,20640.000000,20640.000000
mean,-119.569704,35.631861,28.639486,2635.763081,537.870553,1425.476744,499.539680,3.870671,206855.816909
std,2.003532,2.135952,12.585558,2181.615252,421.385070,1132.462122,382.329753,1.899822,115395.615874
min,-124.350000,32.540000,1.000000,2.000000,1.000000,3.000000,1.000000,0.499900,14999.000000
25%,-121.800000,33.930000,18.000000,1447.750000,296.000000,787.000000,280.000000,2.563400,119600.000000
50%,-118.490000,34.260000,29.000000,2127.000000,435.000000,1166.000000,409.000000,3.534800,179700.000000
75%,-118.010000,37.710000,37.000000,3148.000000,647.000000,1725.000000,605.000000,4.743250,264725.000000
max,-114.310000,41.950000,52.000000,39320.000000,6445.000000,35682.000000,6082.000000,15.000100,500001.000000


In [5]:
df.isnull().sum() / len(df)

,0
longitude,0.000000
latitude,0.000000
housing_median_age,0.000000
total_rooms,0.000000
total_bedrooms,0.010029
population,0.000000
households,0.000000
median_income,0.000000
median_house_value,0.000000
ocean_proximity,0.000000


In [6]:
df = df.dropna()
# usuniecie braków, gdzie wystepują

In [7]:
df.isnull().sum() / len(df)

,0
longitude,0.000000
latitude,0.000000
housing_median_age,0.000000
total_rooms,0.000000
total_bedrooms,0.000000
population,0.000000
households,0.000000
median_income,0.000000
median_house_value,0.000000
ocean_proximity,0.000000


In [8]:
df.describe(include=['object'])

,ocean_proximity
count,20433
unique,5
top,<1H OCEAN
freq,9034


In [9]:
df.ocean_proximity.value_counts()

,count
ocean_proximity,
<1H OCEAN,9034
INLAND,6496
NEAR OCEAN,2628
NEAR BAY,2270
ISLAND,5


In [10]:
# wykres korelacji
px.scatter(df, x="median_income", y="median_house_value", title="Income vs House Value")

In [11]:
# rozkład zmiennej docelowej median_house_value
px.histogram(df, x='median_house_value')

In [12]:
df.median_house_value.value_counts()

,count
median_house_value,
500001.000000,958
137500.000000,119
162500.000000,116
112500.000000,103
187500.000000,92
...,...
321700.000000,1
300800.000000,1
393100.000000,1


In [13]:
# pozostawienie tylko normalnych cen domów, usuniecie sztucznych 500001
df = df[df["median_house_value"] < 500001]

In [14]:
px.histogram(df, x='median_house_value')

In [15]:
df.median_house_value.value_counts()

,count
median_house_value,
137500.000000,119
162500.000000,116
112500.000000,103
187500.000000,92
225000.000000,91
...,...
46200.000000,1
385200.000000,1
358200.000000,1


In [16]:
# zakodowanie kategorycznych kategorii(teskstowych)
df = pd.get_dummies(df, drop_first=True)
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity_INLAND,ocean_proximity_ISLAND,ocean_proximity_NEAR BAY,ocean_proximity_NEAR OCEAN
0,-122.230000,37.880000,41.000000,880.000000,129.000000,322.000000,126.000000,8.325200,452600.000000,False,False,True,False
1,-122.220000,37.860000,21.000000,7099.000000,1106.000000,2401.000000,1138.000000,8.301400,358500.000000,False,False,True,False
2,-122.240000,37.850000,52.000000,1467.000000,190.000000,496.000000,177.000000,7.257400,352100.000000,False,False,True,False
3,-122.250000,37.850000,52.000000,1274.000000,235.000000,558.000000,219.000000,5.643100,341300.000000,False,False,True,False
4,-122.250000,37.850000,52.000000,1627.000000,280.000000,565.000000,259.000000,3.846200,342200.000000,False,False,True,False


In [17]:
# Zamiana bool na int 0/1
bool_cols = df.select_dtypes(include='bool').columns

df[bool_cols] = df[bool_cols].astype(int)

In [18]:
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity_INLAND,ocean_proximity_ISLAND,ocean_proximity_NEAR BAY,ocean_proximity_NEAR OCEAN
0,-122.230000,37.880000,41.000000,880.000000,129.000000,322.000000,126.000000,8.325200,452600.000000,0,0,1,0
1,-122.220000,37.860000,21.000000,7099.000000,1106.000000,2401.000000,1138.000000,8.301400,358500.000000,0,0,1,0
2,-122.240000,37.850000,52.000000,1467.000000,190.000000,496.000000,177.000000,7.257400,352100.000000,0,0,1,0
3,-122.250000,37.850000,52.000000,1274.000000,235.000000,558.000000,219.000000,5.643100,341300.000000,0,0,1,0
4,-122.250000,37.850000,52.000000,1627.000000,280.000000,565.000000,259.000000,3.846200,342200.000000,0,0,1,0


In [19]:
X = df.drop("median_house_value", axis=1)
y = df["median_house_value"]

In [20]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [21]:
print(f'X_train shape: {X_train.shape}')
print(f'y_train shape: {y_train.shape}')
print(f'X_test shape: {X_test.shape}')
print(f'y_test shape: {y_test.shape}')

X_train shape: (15580, 12)
y_train shape: (15580,)
X_test shape: (3895, 12)
y_test shape: (3895,)


In [22]:
X_train.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,ocean_proximity_INLAND,ocean_proximity_ISLAND,ocean_proximity_NEAR BAY,ocean_proximity_NEAR OCEAN
5148,-118.270000,33.970000,34.000000,1462.000000,394.000000,1310.000000,351.000000,1.155700,0,0,0,0
1378,-122.100000,38.020000,28.000000,4308.000000,824.000000,2086.000000,776.000000,3.652300,0,0,1,0
10567,-117.740000,33.730000,18.000000,328.000000,68.000000,391.000000,60.000000,4.116700,0,0,0,0
16253,-121.270000,37.960000,52.000000,583.000000,114.000000,310.000000,93.000000,2.562500,1,0,0,0
3920,-118.560000,34.200000,36.000000,1544.000000,308.000000,891.000000,286.000000,4.175000,0,0,0,0


In [23]:
y_train.head()

,median_house_value
5148,90100.000000
1378,159700.000000
10567,87500.000000
16253,54200.000000
3920,190900.000000


standaryzacja

In [24]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [25]:
y_scaler = StandardScaler()

y_train = y_scaler.fit_transform(y_train.values.reshape(-1,1))
y_test = y_scaler.transform(y_test.values.reshape(-1,1))

In [26]:
X_train

array([[ 0.652101226036, -0.788944225811,  0.452039473674, ..., -0.017917234366, -0.35181501198 , -0.372789682109],
       [-1.257708150263,  1.095295386106, -0.027724485533, ..., -0.017917234366,  2.842402870681, -0.372789682109],
       [ 0.916382941294, -0.90060286948 , -0.827331084211, ..., -0.017917234366, -0.35181501198 , -0.372789682109],
       ...,
       [ 0.667060568409, -0.663328251683,  0.21215749407 , ..., -0.017917234366, -0.35181501198 , -0.372789682109],
       [-1.212830123144,  0.88128298574 , -1.786859002625, ..., -0.017917234366,  2.842402870681, -0.372789682109],
       [-0.539659716354, -0.216693677007, -1.22713438355 , ..., -0.017917234366, -0.35181501198 ,  2.682477675732]])

In [27]:
df[
[
'ocean_proximity_INLAND',
'ocean_proximity_ISLAND',
'ocean_proximity_NEAR BAY',
'ocean_proximity_NEAR OCEAN'
]
].sum()

,0
ocean_proximity_INLAND,6469
ocean_proximity_ISLAND,5
ocean_proximity_NEAR BAY,2077
ocean_proximity_NEAR OCEAN,2419


In [28]:
import tensorflow as tf

def r2_score(y_true, y_pred):
    ss_res = tf.reduce_sum(tf.square(y_true - y_pred))
    ss_tot = tf.reduce_sum(tf.square(y_true - tf.reduce_mean(y_true)))
    return 1 - ss_res / (ss_tot + tf.keras.backend.epsilon())

budowa modelu - prosty model

In [29]:
model = Sequential()
model.add(Dense(64, activation='relu', input_shape=(X_train.shape[1],)))
model.add(Dense(32, activation='relu'))
model.add(Dense(1))

model.compile(optimizer='adam', loss='mse', metrics=['mae', 'mse',r2_score])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning:

Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.



In [31]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [32]:
history = model.fit(X_train, y_train, epochs=30, validation_split=0.2, verbose=1, batch_size=32)

Epoch 1/30
390/390 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 0.4260 - mae: 0.4677 - mse: 0.4260 - r2_score: 0.5599 - val_loss: 0.3265 - val_mae: 0.4123 - val_mse: 0.3265 - val_r2_score: 0.6495
Epoch 2/30
390/390 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.3418 - mae: 0.4130 - mse: 0.3418 - r2_score: 0.6435 - val_loss: 0.3065 - val_mae: 0.3896 - val_mse: 0.3065 - val_r2_score: 0.6707
Epoch 3/30
390/390 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.3153 - mae: 0.3982 - mse: 0.3153 - r2_score: 0.6689 - val_loss: 0.3061 - val_mae: 0.3974 - val_mse: 0.3061 - val_r2_score: 0.6677
Epoch 4/30
390/390 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.3210 - mae: 0.3924 - mse: 0.3210 - r2_score: 0.6620 - val_loss: 0.2984 - val_mae: 0.3940 - val_mse: 0.2984 - val_r2_score: 0.6762
Epoch 5/30
390/390 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.2968 - mae: 0.3842 - mse: 0.2968 - r2_score: 0.6903 - val_loss: 0.2884 - val_mae: 0.3836 - val_mse: 0.2884 - val_r2_score: 0.6895
Epoch 6/30
390/390 ━━━━━━━━━━━━━━━━━━━━ 

In [33]:
def plot_hist(history):
    hist = pd.DataFrame(history.history)
    hist['epoch'] = history.epoch
    hist['rmse'] = np.sqrt(hist['mse'])

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=hist['epoch'], y=hist['mae'], name='mae', mode='markers+lines'))
    fig.add_trace(go.Scatter(x=hist['epoch'], y=hist['val_mae'], name='val_mae', mode='markers+lines'))
    fig.update_layout(width=1000, height=500, title='MAE vs. VAL_MAE', xaxis_title='Epoki', yaxis_title='Mean Absolute Error', yaxis_type='log')
    fig.show()

plot_hist(history)

In [34]:
test_loss, test_mae, _, _ = model.evaluate(X_test, y_test)

print("Test MSE:", test_loss)
print("Test MAE:", test_mae)

122/122 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.2539 - mae: 0.3516 - mse: 0.2539 - r2_score: 0.7383
Test MSE: 0.2539239525794983
Test MAE: 0.351639986038208


In [35]:
from sklearn.metrics import r2_score

predictions = model.predict(X_test)

r2 = r2_score(y_test, predictions)
print("R2:", r2)

122/122 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
R2: 0.7506675623979435


In [36]:
# poprawa danych - cofniecie skalowania
predictions = model.predict(X_test).flatten()

# cofnięcie skalowania
predictions_real = y_scaler.inverse_transform(predictions.reshape(-1,1)).flatten()
y_test_real = y_scaler.inverse_transform(y_test).flatten()

results = pd.DataFrame({
  "rzeczywiste": y_test_real,
  "przewidywane": predictions_real,
   "błąd": abs(predictions_real - y_test_real)
})

results.head()

122/122 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step


,rzeczywiste,przewidywane,błąd
0,96200.000000,104314.367188,8114.367188
1,178700.000000,196943.250000,18243.250000
2,60600.000000,68090.320312,7490.320312
3,191900.000000,176061.703125,15838.296875
4,51000.000000,61591.539062,10591.539062


In [37]:
fig = px.scatter(results, x='rzeczywiste', y='przewidywane')
fig.add_trace(go.Scatter(x=[0, 500000], y=[0, 500000], mode='lines'))
fig.show()

In [38]:
px.histogram(results, x='błąd', marginal='rug', width=1000)

ulepszony model: BatchNormalization(), Dropout(), earlyStopping(), learning_rate

In [39]:
from tensorflow.keras.regularizers import l2
model = Sequential()
model.add(Dense(128, activation='relu',kernel_regularizer=l2(0.001), input_shape=(X_train.shape[1],)))
model.add(BatchNormalization())
model.add(Dropout(0.3))
model.add(Dense(64,kernel_regularizer=l2(0.001), activation='relu'))
model.add(BatchNormalization())
model.add(Dropout(0.2))
model.add(Dense(32,kernel_regularizer=l2(0.001), activation='relu'))
model.add(BatchNormalization())
model.add(Dense(1))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning:

Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.



In [40]:
from tensorflow.keras.callbacks import ReduceLROnPlateau
lr_scheduler = ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6)

In [41]:
# interpretacja wskażnika R2
def r2_score(y_true, y_pred):
    ss_res = tf.reduce_sum(tf.square(y_true - y_pred))
    ss_tot = tf.reduce_sum(tf.square(y_true - tf.reduce_mean(y_true)))
    return 1 - ss_res / (ss_tot + tf.keras.backend.epsilon())

In [42]:
model.compile(optimizer=tf.keras.optimizers.AdamW(learning_rate=0.001, weight_decay=1e-4), loss='mse', metrics=['mae', 'mse',r2_score])

In [43]:
early_stop = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)

In [44]:
history = model.fit(X_train, y_train, epochs=100, validation_split=0.2, verbose=1, batch_size=32, callbacks=[early_stop, lr_scheduler])

Epoch 1/100
390/390 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - loss: 0.8579 - mae: 0.6311 - mse: 0.7106 - r2_score: 0.2381 - val_loss: 0.5377 - val_mae: 0.4848 - val_mse: 0.3928 - val_r2_score: 0.5780 - learning_rate: 0.0010
Epoch 2/100
390/390 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.6140 - mae: 0.5115 - mse: 0.4723 - r2_score: 0.4921 - val_loss: 0.4675 - val_mae: 0.4166 - val_mse: 0.3292 - val_r2_score: 0.6494 - learning_rate: 0.0010
Epoch 3/100
390/390 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.5518 - mae: 0.4784 - mse: 0.4174 - r2_score: 0.5576 - val_loss: 0.4425 - val_mae: 0.3985 - val_mse: 0.3123 - val_r2_score: 0.6645 - learning_rate: 0.0010
Epoch 4/100
390/390 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - loss: 0.5149 - mae: 0.4577 - mse: 0.3893 - r2_score: 0.5872 - val_loss: 0.4218 - val_mae: 0.3913 - val_mse: 0.3007 - val_r2_score: 0.6787 - learning_rate: 0.0010
Epoch 5/100
390/390 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.4903 - mae: 0.4464 - mse: 0.3740 - r2_score: 0.6019 - val_loss: 0.408

In [45]:
def plot_hist(history):
    hist = pd.DataFrame(history.history)
    hist['epoch'] = history.epoch
    hist['rmse'] = np.sqrt(hist['mse'])

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=hist['epoch'], y=hist['mae'], name='mae', mode='markers+lines'))
    fig.add_trace(go.Scatter(x=hist['epoch'], y=hist['val_mae'], name='val_mae', mode='markers+lines'))
    fig.update_layout(width=1000, height=500, title='MAE vs. VAL_MAE', xaxis_title='Epoki', yaxis_title='Mean Absolute Error', yaxis_type='log')
    fig.show()

plot_hist(history)

In [46]:
test_loss, test_mae, _, _ = model.evaluate(X_test, y_test)

print("Test MSE:", test_loss)
print("Test MAE:", test_mae)

122/122 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.2349 - mae: 0.3175 - mse: 0.2181 - r2_score: 0.7727
Test MSE: 0.23492148518562317
Test MAE: 0.31746605038642883


In [47]:
from sklearn.metrics import r2_score

predictions = model.predict(X_test)

r2 = r2_score(y_test, predictions)
print("R2:", r2)

122/122 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
R2: 0.7858785465312974


In [48]:
# poprawa danych - cofniecie skalowania
predictions = model.predict(X_test).flatten()

# cofnięcie skalowania
predictions_real = y_scaler.inverse_transform(predictions.reshape(-1,1)).flatten()
y_test_real = y_scaler.inverse_transform(y_test).flatten()

results = pd.DataFrame({
    "rzeczywiste": y_test_real,
    "przewidywane": predictions_real,
    "błąd": abs(predictions_real - y_test_real)
})

results.head()

122/122 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step


,rzeczywiste,przewidywane,błąd
0,96200.000000,112501.140625,16301.140625
1,178700.000000,203692.093750,24992.093750
2,60600.000000,74308.039062,13708.039062
3,191900.000000,178050.750000,13849.250000
4,51000.000000,56234.523438,5234.523438


In [49]:
fig = px.scatter(results, x='rzeczywiste', y='przewidywane')
fig.add_trace(go.Scatter(x=[0, 500000], y=[0, 500000], mode='lines'))
fig.show()

RandomForestRegressor

In [50]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import (mean_squared_error, mean_absolute_error, r2_score)

In [51]:
rf = RandomForestRegressor(random_state=42)

In [52]:
param_dist = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [10, 20, 30, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"]
}

In [53]:
random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions = param_dist,
    n_iter = 20, # liczba losowych kombinacji
    cv = 5,
    scoring="r2",
    verbose=2,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train) # 5 * 20 = 100 kombinacji

Fitting 5 folds for each of 20 candidates, totalling 100 fits


/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning:

A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.

/usr/local/lib/python3.12/dist-packages/sklearn/base.py:1389: DataConversionWarning:

A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().



RandomizedSearchCV(cv=5, estimator=RandomForestRegressor(random_state=42),
                   n_iter=20, n_jobs=-1,
                   param_distributions={'max_depth': [10, 20, 30, None],
                                        'max_features': ['sqrt', 'log2'],
                                        'min_samples_leaf': [1, 2, 4],
                                        'min_samples_split': [2, 5, 10],
                                        'n_estimators': [100, 200, 300, 500]},
                   random_state=42, scoring='r2', verbose=2)

In [54]:
print("Najlepsze parametry:", random_search.best_params_)
print("Najlepszy wynik R2:", random_search.best_score_)

Najlepsze parametry: {'n_estimators': 200, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': 30}
Najlepszy wynik R2: 0.7768543200019292


In [55]:
best_rf = random_search.best_estimator_

predictions_rf = best_rf.predict(X_test)

In [56]:
print("MSE:", mean_squared_error(y_test, predictions_rf))
print("MAE:", mean_absolute_error(y_test, predictions_rf))
print("R2:", r2_score(y_test, predictions_rf))

MSE: 0.21286996075827266
MAE: 0.3146792244665789
R2: 0.7909791686337703


In [57]:
predictions = model.predict(X_test)

# cofnięcie skalowania y
predictions = y_scaler.inverse_transform(predictions).flatten()
y_true = y_scaler.inverse_transform(y_test).flatten()

results = pd.DataFrame({
    "rzeczywiste": y_true,
    "przewidywane": predictions,
    "błąd": predictions - y_true
})

fig = px.scatter(results, x="rzeczywiste", y="przewidywane")

fig.add_trace(
    go.Scatter(
        x=[0, 500000],
        y=[0, 500000],
        mode="lines",
        name="Idealne dopasowanie",
        line=dict(color="red")
    )
)

fig.show()

122/122 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step






# Porównanie modeli

| Model                 | MAE   | MSE   | R^2    |
|----------------------|-------|-------|-------|
| Prosta sieć neuronowa | 0.3516 | 0.2539 | 0.7507 |
| Ulepszona sieć NN     | 0.3175 | 0.2349 | 0.7859 |
| RandomForest          | 0.3147 | 0.2129 | 0.7910 |

Wraz z rozbudową architektury sieci neuronowej następowała stopniowa poprawa jakości predykcji.

Dodanie Batch Normalization, Dropout, regularyzacji L2 oraz Early Stopping zmniejszyło błąd predykcji i poprawiło zdolność modelu do uogólniania.

Nie zaobserwowano przeuczenia, ponieważ zarówno błąd treningowy, jak i walidacyjny zmniejszały się równolegle.

Najlepsze wyniki uzyskał jednak model Random Forest.

Uzyskany współczynnik R^2 = 0.791 oznacza, że model wyjaśnia około 79% zmienności cen mieszkań występujących w zbiorze danych.

Lepszy wynik Random Forest wynika z faktu, że dane mają charakter tabelaryczny. Modele drzewiaste bardzo dobrze odnajdują nieliniowe zależności oraz interakcje pomiędzy cechami, nie wymagając przy tym skomplikowanego procesu uczenia charakterystycznego dla sieci neuronowych.

Wnioski

Projekt pokazał pełny proces budowy modeli regresyjnych:
- przygotowanie danych,
- budowę podstawowej sieci neuronowej,
- zastosowanie technik poprawiających jakość uczenia,
- analizę metryk jakości,
- ocenę zjawiska overfittingu,
- porównanie z klasycznym modelem uczenia maszynowego,
- strojenie hiperparametrów.